# Arrhythmia Detection with 1D-CNN -- Full-Scheme Training Notebook (Kaggle)

Single end-to-end notebook covering the whole research pipeline:

1. **Preprocessing** -- polyphase FIR resampling + wavelet `db4` cleaning to a unified **250 Hz** representation for both PTB-XL and Chapman (auto-skips if preprocessed data is already present).
2. **Training** -- every *scheme* in the research matrix:
   - classification head: **softmax** (multiclass) and **sigmoid** (multilabel)
   - label scheme: **mapped** (shared 4-class) and **native** (dataset codes)
   - dataset matrix: PTB-XL / Chapman in all valid train-test combinations
3. **Post-processing** -- per-experiment confusion matrices, classification reports, misclassified ECG plots, **zero-shot cross-dataset evaluation**, statistical significance tests, and a single results archive.

Every result is persisted under `output/research_experiments/` (master tracker, per-experiment folders, cross-domain summaries) and packed into `output/kaggle_all_results.zip`.

> Use **fast mode** (default) to smoke-test every scheme end-to-end quickly, then switch to **full mode** for the reference configuration from `docs/training-configs/`.


## How to run on Kaggle

1. Create a Kaggle notebook with **GPU** accelerator (P100 or T4). TensorFlow is preinstalled.
2. Make the repo source available so `src/config` exists in the workspace:
   - either clone/sync the project into `/kaggle/working/arrhythmia-detection-1d-cnn`, or
   - open this notebook inside a copy of the project on your local machine and use *Kaggle -> File -> Import -> Notebook*.
3. Provide the data in one of two ways (this notebook auto-detects which one you have):
   - **Preprocessed layout** (recommended, uploads faster): a `dataset/` folder mirroring the local layout
     `dataset/PTBXL/ptbxl_database.csv`, `dataset/resample/manifest_{ptbxl,chapman}.csv`,
     `dataset/resample/cleaned_100_to_250`, `dataset/resample/cleaned_500_to_250`,
     `dataset/resample/chapman_clean_500_to_250`.
     Upload it as a Kaggle Dataset and it is mounted at `/kaggle/input/.../dataset`.
     *The PTB-XL database CSV is only required when the `native` PTB-XL plans are enabled.*
   - **Raw layout**: `dataset/PTBXL/{ptbxl_database.csv, scp_statements.csv, records100, records500}`
     and `dataset/Chapman/{ConditionNames_SNOMED-CT.csv, WFDBRecords}`. Preprocessing runs from scratch.
4. Run all cells. Training duration depends on `FAST_MODE` (see the training cells).
5. Download `output/kaggle_all_results.zip` from the last cell.


In [ ]:
import os, sys, subprocess

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# wfdb + DSP + stats extras (TF / numpy / pandas / sklearn already on Kaggle)
_pip("wfdb", "PyWavelets", "imbalanced-learn", "fastdtw", "neurokit2", "seaborn", "tabulate")

import tensorflow as tf
print("tensorflow", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
import os, sys

def _find_repo_root():
    cwd = os.getcwd()
    for cand in [cwd, os.path.join(cwd, "arrhythmia-detection-1d-cnn"), os.path.dirname(cwd)]:
        if os.path.isdir(os.path.join(cand, "src", "config")):
            return os.path.abspath(cand)
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None:
    raise RuntimeError(
        "Repo source not found. Clone/sync the project into /kaggle/working "
        "(so that src/config exists) before running this notebook."
    )
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
import os, glob, shutil

from src.config import config as cfg
from src.config.experiment_configs import Config

def _has(p):
    return os.path.exists(p)

def _populated(d):
    if not _has(d):
        return False
    try:
        return len([f for f in os.listdir(d) if f.endswith(".npy")]) > 0
    except OSError:
        return False

# ----------------------------------------------------------------------
# 1) If a Kaggle input mount provides a `dataset/` folder, copy it in.
# ----------------------------------------------------------------------
mounts = sorted(glob.glob("/kaggle/input/*/dataset"))
if mounts and not _populated(cfg.SUB_FOLDERS["E2_clean_100_to_250"]):
    src = mounts[0]
    print("Mounting dataset from Kaggle input:", src)
    for sub in ["PTBXL", "Chapman", "resample"]:
        p = os.path.join(src, sub)
        if os.path.isdir(p):
            shutil.copytree(p, os.path.join(cfg.DATASET_DIR, sub), dirs_exist_ok=True)

# ----------------------------------------------------------------------
# 2) Readiness report
# ----------------------------------------------------------------------
PTB_MAN     = os.path.join(cfg.RESAMPLE_BASE, "manifest_ptbxl.csv")
CHAP_MAN    = os.path.join(cfg.RESAMPLE_BASE, "manifest_chapman.csv")
PTB_SIGNAL  = cfg.SUB_FOLDERS["E2_clean_100_to_250"]
CHAP_SIGNAL = cfg.SUB_FOLDERS["Chapman_clean_500_to_250"]

PTB_READY  = _has(PTB_MAN) and _populated(PTB_SIGNAL)
CHAP_READY = _has(CHAP_MAN) and _populated(CHAP_SIGNAL)

RAW_PTB  = _has(cfg.PTBXL_CSV) and len(glob.glob(os.path.join(cfg.DATASET_DIR, "PTBXL", "records*"))) > 0
RAW_CHAP = _has(cfg.CHAPMAN_RECS)

print("Preprocessed PTB-XL  :", PTB_READY)
print("Preprocessed Chapman :", CHAP_READY)
print("Raw PTB-XL present   :", RAW_PTB)
print("Raw Chapman present  :", RAW_CHAP)

if not PTB_READY and not RAW_PTB:
    print("WARNING: PTB-XL needs preprocessed dataset/resample OR raw dataset/PTBXL.")
if not CHAP_READY and not RAW_CHAP:
    print("WARNING: Chapman needs preprocessed dataset/resample OR raw dataset/Chapman.")


In [ ]:
import time, runpy

t0 = time.time()

if PTB_READY:
    print("PTB-XL already preprocessed - skipping.")
elif RAW_PTB:
    print("Preprocessing PTB-XL (100Hz + 500Hz -> 250Hz pipelines)...")
    runpy.run_module("src.preprocessing.proccess_ptbxl", run_name="__main__")
    print("PTB-XL preprocessing finished in %.1f min" % ((time.time() - t0) / 60))
else:
    raise RuntimeError("PTB-XL data unavailable (neither preprocessed nor raw).")


In [ ]:
t0 = time.time()

if CHAP_READY:
    print("Chapman already preprocessed - skipping.")
elif RAW_CHAP:
    print("Preprocessing Chapman (unified 250Hz pipeline)...")
    runpy.run_module("src.preprocessing.proccess_chapman", run_name="__main__")
    print("Chapman preprocessing finished in %.1f min" % ((time.time() - t0) / 60))
else:
    raise RuntimeError("Chapman data unavailable (neither preprocessed nor raw).")


In [ ]:
# Optional EDA / preprocessing-quality audit.
# Both are slow on full data; keep False unless you need the audit figures.
RUN_EDA = False

if RUN_EDA:
    runpy.run_module("src.analysis.eda_visualization", run_name="__main__")
    runpy.run_module("src.analysis.eda_quantitative_audit", run_name="__main__")
    print("EDA finished.")
else:
    print("EDA skipped (set RUN_EDA = True to enable).")


## The scheme matrix

| Head      | Label scheme | Valid dataset pairs |
|-----------|--------------|---------------------|
| softmax   | mapped       | PTBXL->PTBXL, PTBXL->CHAPMAN, CHAPMAN->PTBXL, CHAPMAN->CHAPMAN |
| softmax   | native       | PTBXL->PTBXL, CHAPMAN->CHAPMAN |
| sigmoid   | mapped       | PTBXL->PTBXL, PTBXL->CHAPMAN, CHAPMAN->PTBXL, CHAPMAN->CHAPMAN |
| sigmoid   | native       | PTBXL->PTBXL, CHAPMAN->CHAPMAN |

Cross-dataset runs (**mapped** only) are zero-shot external validations; **native**
labels are dataset-specific codes (PTB-XL SCP / Chapman SNOMED-CT) and only make
sense per dataset. That gives **12 plans total**.


In [ ]:
# Full scheme matrix evaluated by this notebook.
MAPPED_PAIRS = [("PTBXL", "PTBXL"), ("PTBXL", "CHAPMAN"),
                ("CHAPMAN", "PTBXL"), ("CHAPMAN", "CHAPMAN")]
NATIVE_PAIRS = [("PTBXL", "PTBXL"), ("CHAPMAN", "CHAPMAN")]

PLANS = []
for scheme in ["softmax", "sigmoid"]:
    for label_scheme, pairs in [("mapped", MAPPED_PAIRS), ("native", NATIVE_PAIRS)]:
        for train_ds, test_ds in pairs:
            PLANS.append({
                "scheme": scheme,
                "label_scheme": label_scheme,
                "train": train_ds,
                "test": test_ds,
                "name": "%s_%s_%s_to_%s" % (scheme, label_scheme, train_ds, test_ds),
            })

# Restrict here if you only want a subset, e.g. PLANS_TO_RUN = PLANS[:4]
PLANS_TO_RUN = PLANS

for p in PLANS_TO_RUN:
    print(p["name"])
print("Total plans:", len(PLANS_TO_RUN))


In [ ]:
# FAST_MODE renders every scheme quickly (few epochs, single pipeline, small
# architecture) so the notebook can run end-to-end on Kaggle. Set FAST_MODE =
# False for the full reference configuration (docs/training-configs).
FAST_MODE = True

if FAST_MODE:
    Config.EPOCHS = 5
    Config.ACTIVE_DATASETS = ["E2_clean_100_to_250"]
    Config.FILTER_SPACES = {"Small": [32, 64, 128, 128, 256]}
    Config.KERNEL_SPACES = {"Balanced": [15, 11, 7, 5, 3]}
    Config.DILATION_SPACES = {"Progressive_Dilation": [1, 2, 4, 8, 16]}
    Config.TEMPORAL_MODELS = ["Pure_CNN"]
    Config.UNDERSAMPLE_RATIO = None
    Config.OVERSAMPLE_METHOD = None
else:
    Config.EPOCHS = 35
    Config.ACTIVE_DATASETS = ["E2_clean_100_to_250", "E3_clean_500_to_250"]
    Config.FILTER_SPACES = {"Medium": [64, 128, 256, 256, 512]}
    Config.KERNEL_SPACES = {"Balanced": [15, 11, 7, 5, 3]}
    Config.DILATION_SPACES = {"Progressive_Dilation": [1, 2, 4, 8, 16]}
    Config.TEMPORAL_MODELS = ["Pure_CNN"]
    Config.UNDERSAMPLE_RATIO = 10
    Config.OVERSAMPLE_METHOD = "smote_tomek"
    Config.OVERSAMPLE_STRATEGY = {"AF": 3000, "Bradikardia": 3000, "Takikardia": 3000}

print("FAST_MODE =", FAST_MODE, "| epochs =", Config.EPOCHS,
      "| pipelines =", Config.ACTIVE_DATASETS)


## Training: all schemes

Each plan drives the unified runner (`src/experiments/run_experiment.py`) through
the `Config` object, then the notebook captures every new row appended to the
master tracker. Outputs per experiment (config JSON, training log, best model,
confusion matrix, classification report, misclassified plots) are saved by the
runner itself under `output/research_experiments/`.


In [ ]:
import runpy, gc, os
import pandas as pd
import tensorflow as tf

RESULT_ROOT = os.path.join(cfg.OUTPUT_DIR, "research_experiments")
TRACKER = os.path.join(RESULT_ROOT, Config.MASTER_TRACKER_CSV)
PLAN_SUMMARY = os.path.join(RESULT_ROOT, "plan_summary.csv")

def _set_config(plan):
    Config.SCHEME = plan["scheme"]
    Config.LABEL_SCHEME = plan["label_scheme"]
    Config.TRAIN_DATASET = plan["train"]
    Config.TEST_DATASET = plan["test"]

def _read_tracker():
    if not os.path.exists(TRACKER):
        return pd.DataFrame()
    return pd.read_csv(TRACKER)

def _write_plan_log(rows):
    pd.DataFrame(rows).to_csv(PLAN_SUMMARY, index=False)

base = _read_tracker()
known = set(base["Experiment"].astype(str)) if len(base) else set()
plan_rows = []

for plan in PLANS_TO_RUN:
    print("\n" + "=" * 70)
    print("PLAN: %s  |  %s / %s  |  train=%s test=%s"
          % (plan["name"], plan["scheme"], plan["label_scheme"],
             plan["train"], plan["test"]))
    print("=" * 70)

    _set_config(plan)

    row = dict(plan)
    try:
        runpy.run_module("src.experiments.run_experiment", run_name="__main__")
        tf.keras.backend.clear_session()
        gc.collect()

        now = _read_tracker()
        new = now[~now["Experiment"].astype(str).isin(known)] if len(now) else now
        known = set(now["Experiment"].astype(str)) if len(now) else set()

        if len(new):
            top = new.sort_values("Macro_F1", ascending=False).iloc[0]
            row.update({
                "runs": int(len(new)),
                "best_Macro_F1": float(top["Macro_F1"]),
                "best_Balanced_Accuracy": float(top["Balanced_Accuracy"]),
                "error": "",
            })
        else:
            row.update({"runs": 0, "best_Macro_F1": float("nan"),
                        "best_Balanced_Accuracy": float("nan"), "error": "no rows"})
    except Exception as e:
        row.update({"runs": 0, "best_Macro_F1": float("nan"),
                    "best_Balanced_Accuracy": float("nan"), "error": str(e)})
        print("PLAN FAILED:", plan["name"], "->", e)

    plan_rows.append(row)
    _write_plan_log(plan_rows)
    print("Recorded:", row["name"], "runs =", row["runs"],
          "| best F1 =", row["best_Macro_F1"])

print("\nAll plans finished. Per-plan log:", PLAN_SUMMARY)


## Post-processing: cross-dataset (zero-shot) evaluation

Every trained **mapped** 4-class model is evaluated without retraining against the
test split of the other dataset (PTB-XL models on Chapman and vice versa). Native
models are skipped here because their label spaces do not overlap. Reports and
confusion matrices are written under `output/research_experiments/cross_domain/`.


In [ ]:
import os, json, glob
import pandas as pd
import tensorflow as tf

from src.evaluation import cross_dataset_test as xd

CROSS_ROOT = os.path.join(RESULT_ROOT, "cross_domain")

# Mapped test splits of both datasets (zero-shot evaluation targets).
EVAL_TARGETS = {
    "CHAPMAN": xd.load_target_eval("CHAPMAN"),
    "PTBXL": xd.load_target_eval("PTBXL"),
}

rows = []

for exp_dir in sorted(glob.glob(os.path.join(RESULT_ROOT, "*", "*"))):
    model_file = xd.resolve_model_file(exp_dir)
    cfg_file = os.path.join(exp_dir, "experiment_config.json")
    if model_file is None or not os.path.exists(cfg_file):
        continue
    with open(cfg_file, encoding="utf-8") as f:
        meta = json.load(f)
    names = meta.get("class_names") or []
    # Cross-dataset comparison is only meaningful for the shared 4-class (mapped) models.
    if len(names) != 4:
        print("Skip (native labels, not comparable):", os.path.basename(exp_dir))
        continue
    is_sigmoid = str(meta.get("scheme", "softmax")).lower() == "sigmoid"
    thresholds = xd.load_sigmoid_thresholds(exp_dir) if is_sigmoid else None

    for target, (X, y_names, _cn) in EVAL_TARGETS.items():
        try:
            out = xd.evaluate_model(
                model_file,
                "sigmoid" if is_sigmoid else "softmax",
                X, y_names, names, thresholds
            )
        except Exception as e:
            print("Error", os.path.basename(exp_dir), "->", e)
            continue
        rows.append({
            "Experiment": os.path.basename(exp_dir),
            "Scheme": meta.get("scheme"),
            "Label_Scheme": meta.get("label_scheme"),
            "Train_Dataset": meta.get("train_dataset"),
            "Test_Dataset": target,
            "Accuracy": float(out["Accuracy"]),
            "Macro_Recall": float(out["Macro_Recall"]),
            "Macro_F1": float(out["Macro_F1"]),
        })

        out_dir = os.path.join(CROSS_ROOT,
                               "%s_to_%s" % (meta.get("train_dataset", "?"), target),
                               os.path.basename(exp_dir))
        os.makedirs(out_dir, exist_ok=True)
        with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
            f.write(out["report"])
        out["cm_fig"].savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
        tf.keras.backend.clear_session()

cross_df = pd.DataFrame(rows)
cross_csv = os.path.join(RESULT_ROOT, "cross_domain_results.csv")
cross_df.to_csv(cross_csv, index=False)
print("\nCross-dataset results:", cross_csv)
if len(cross_df):
    print(cross_df.to_string(index=False))
else:
    print("  (no comparable mapped models found yet)")


In [ ]:
# Statistical significance analysis over the unified master tracker.
runpy.run_module("src.evaluation.run_statistical_tests", run_name="__main__")


## Results summary & download

Everything recorded during this run is shown below and packaged into a single
zip archive for download.


In [ ]:
import os, zipfile
import pandas as pd
from IPython.display import display, Markdown

RESULT_ROOT = os.path.join(cfg.OUTPUT_DIR, "research_experiments")

trk = pd.read_csv(os.path.join(RESULT_ROOT, Config.MASTER_TRACKER_CSV))
display(Markdown("### Master experiment tracker"))
display(trk.sort_values("Macro_F1", ascending=False).reset_index(drop=True))

ps = os.path.join(RESULT_ROOT, "plan_summary.csv")
if os.path.exists(ps):
    display(Markdown("### Per-plan summary"))
    display(pd.read_csv(ps))

csv_cross = os.path.join(RESULT_ROOT, "cross_domain_results.csv")
if os.path.exists(csv_cross):
    display(Markdown("### Cross-domain (zero-shot) results"))
    display(pd.read_csv(csv_cross))

# ----------------------------------------------------------------------
# Package everything for download
# ----------------------------------------------------------------------
ZIP_NAME = os.path.join(cfg.OUTPUT_DIR, "kaggle_all_results.zip")
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["research_experiments", "statistical_tests", "cross_dataset_test"]:
        root = os.path.join(cfg.OUTPUT_DIR, folder)
        if os.path.isdir(root):
            for dp, _, files in os.walk(root):
                for fn in files:
                    p = os.path.join(dp, fn)
                    zf.write(p, os.path.relpath(p, cfg.OUTPUT_DIR))
print("\nAll results zipped to:", ZIP_NAME)


## Where to find the outputs

- **Master tracker**: `output/research_experiments/master_experiment_tracker.csv`
- **Per-experiment folders**: `output/research_experiments/<scheme>_<label_scheme>/<experiment>/`
  (config JSON, `training_log.csv`, `best_model.keras`, `metrics.csv`,
  `confusion_matrix.png`, `classification_report.txt`, `misclassified/` plots)
- **Cross-domain results**: `output/research_experiments/cross_domain_results.csv`
  and report folders under `output/research_experiments/cross_domain/`
- **Statistical tests**: `output/statistical_tests/*.csv`
- **Everything zipped**: `output/kaggle_all_results.zip`

Switch `FAST_MODE = False` (training cell) and re-run cells 9+ for the full
reference configuration from `docs/training-configs/`.
